# Canonical IR + natural-loop hierarchy at full scale

This is a gated, matched experiment for the general hierarchical LLVM CDFG model. The first run retrains the established H0 model on the new canonicalized schema-v3 tensors while ignoring loop nodes. The second run adds natural-loop composition and the hardware-aligned pooling/path prior. Because both runs use the same tensors, split, seed, optimizer, and depth, their paired difference isolates the architectural bundle.

The default two runs consume at most 11 hours (two 330-minute guards) on dual T4s. Run `region_generic` only after a promising result to separate the natural-loop hierarchy from hardware-aligned composition; all three guards total 16.5 hours. Each run is resumable and packaged independently. A gain below roughly two SMAPE points is not thesis-competitive; a gain of five or more is the signal to ablate and then repeat seeds.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ll-hls4ml"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/ll-hls4ml.git"
REPO_REF = "PASTE_FULL_COMMIT_SHA_HERE"
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-hierarchical"
TENSOR_REVISION = "PASTE_FULL_TENSOR_REVISION_HERE"

# Everything large stays in the ephemeral cache, never /kaggle/working.
HF_CACHE_ROOT = Path("/tmp/ll_hls4ml_region_schema3_hf")
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")
os.environ["HF_XET_CACHE"] = str(HF_CACHE_ROOT / "xet")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# Attach the zip from an earlier session here. Leave None initially.
PREVIOUS_RESULTS_ROOT = None
USE_DDP = True
ACTIVE_RUNS = ["matched_h0", "region_hardware"]
TRAIN_BUDGETS = {
    "matched_h0": "330m",
    "region_hardware": "330m",
    "region_generic": "330m",
}
SEED = 42
PATIENCE = 30
MAX_EPOCHS = 400
ARCHIVE_NAME = "ll_hls4ml_region_hierarchy_scale200_results"
KERNEL_TYPES = [
    "2layer", "3layer", "conv1d", "conv2d",
    "dense_latency", "dense_resource", "rule4ml",
]

def assert_commit(value, name):
    assert re.fullmatch(r"[0-9a-fA-F]{40,64}", value), (
        f"{name} must be an immutable full hash; got {value!r}"
    )

assert_commit(REPO_REF, "REPO_REF")
assert_commit(TENSOR_REVISION, "TENSOR_REVISION")
assert set(ACTIVE_RUNS) <= set(TRAIN_BUDGETS)
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator."
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "float32"
print("PyTorch:", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
print("Precision:", PRECISION, "active runs:", ACTIVE_RUNS)

## Immutable code checkout and dependencies

In [ ]:
%pip install -q torch-geometric "huggingface_hub[hf_xet]>=0.32" pyyaml pandas

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert commit == REPO_REF
sys.path.insert(0, str(REPO_DIR / "src"))

from ll_hls4ml.data.tensorize import EMBED_SIZE
from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE, FUNCTION_FEATURE_SIZE, LOOP_FEATURE_SIZE,
    PRAGMA_FEATURE_SIZE,
)
from ll_hls4ml.models.registry import list_models
assert {"hierarchical", "hierarchical_region"} <= set(list_models())

CANONICAL_SPLIT_PATH = (
    REPO_DIR / "artifacts/results/ll_hls4ml_hierarchy_fusion_scale200_results"
    / "hierarchical_scale200_seed42/split_manifest.json"
)
HISTORICAL_H0_PREDICTIONS_PATH = (
    REPO_DIR / "artifacts/results/ll_hls4ml_hierarchy_fusion_scale200_results"
    / "hierarchical_scale200_seed42/predictions.csv"
)
assert CANONICAL_SPLIT_PATH.is_file()
assert HISTORICAL_H0_PREDICTIONS_PATH.is_file()
print("ll-hls4ml commit:", commit)

## Download the exact cohort into `/tmp` only

In [ ]:
from threading import Event, Thread
import huggingface_hub
from huggingface_hub import hf_hub_download, login, snapshot_download
from huggingface_hub.utils import enable_progress_bars

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
enable_progress_bars()
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

index_path = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="labels.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(HF_CACHE_ROOT / "hub"),
))
tensor_index = json.loads(index_path.read_text())
split_manifest = json.loads(CANONICAL_SPLIT_PATH.read_text())
expected_sizes = {"train": 2742, "validation": 569, "test": 593, "exemplar": 886}
actual_sizes = {name: len(rows) for name, rows in split_manifest.items()}
assert actual_sizes == expected_sizes, (actual_sizes, expected_sizes)
tensor_paths = sorted({
    row["tensor_path"] for rows in split_manifest.values() for row in rows
})
missing_index = [path for path in tensor_paths if path not in tensor_index["labels"]]
assert not missing_index, f"Pinned tensor revision lacks cohort paths: {missing_index[:5]}"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST_PATH = CONFIG_DIR / "matched_full_scale_manifest.json"
SPLIT_MANIFEST_PATH.write_text(json.dumps(split_manifest, indent=2))
download_patterns = [
    *[f"{family}/archive_{archive}/*.pt" for family in KERNEL_TYPES for archive in range(1, 9)],
    *[f"exemplar/archive_{archive}/*.pt" for archive in range(1, 10)],
    "labels.json", "vocab.json",
]

def directory_size(path):
    total = 0
    for item in path.rglob("*") if path.exists() else ():
        try:
            total += item.stat().st_size if item.is_file() else 0
        except FileNotFoundError:
            pass
    return total

def heartbeat(stop):
    while not stop.wait(60):
        count = len(list(HF_CACHE_ROOT.rglob("*.pt")))
        print(f"[download] {count} tensors, {directory_size(HF_CACHE_ROOT) / 2**30:.2f} GiB cached", flush=True)

stop = Event()
thread = Thread(target=heartbeat, args=(stop,), daemon=True)
thread.start()
try:
    TENSOR_DIR = Path(snapshot_download(
        repo_id=TENSOR_REPO_ID, repo_type="dataset", revision=TENSOR_REVISION,
        token=hf_token, cache_dir=str(HF_CACHE_ROOT / "hub"),
        allow_patterns=download_patterns,
    ))
finally:
    stop.set()
    thread.join()

assert not TENSOR_DIR.is_relative_to(WORK), TENSOR_DIR
VOCAB_PATH = TENSOR_DIR / "vocab.json"
assert VOCAB_PATH.is_file()
print("huggingface_hub:", huggingface_hub.__version__)
print("Cohort:", actual_sizes)
print("Tensor cache (outside working):", TENSOR_DIR)

## Fail closed on schema and provenance

In [ ]:
missing_tensors = [path for path in tensor_paths if not (TENSOR_DIR / path).is_file()]
assert not missing_tensors, f"Missing downloaded tensors: {missing_tensors[:5]}"

sample = torch.load(TENSOR_DIR / tensor_paths[0], map_location="cpu", weights_only=False)
assert sample.hierarchy_schema_version == 3
assert sample.llvm_canonicalization_passes == "sroa,mem2reg"
assert sample["function"].x.shape[1] == FUNCTION_FEATURE_SIZE
assert sample["block"].x.shape[1] == BLOCK_FEATURE_SIZE
assert sample["pragma"].x.shape[1] == PRAGMA_FEATURE_SIZE
for node_type in ("variable", "constant"):
    assert sample[node_type].x.shape[1] == EMBED_SIZE

loop_sample = None
for path in tensor_paths:
    candidate = torch.load(TENSOR_DIR / path, map_location="cpu", weights_only=False)
    assert candidate.hierarchy_schema_version == 3
    assert candidate.llvm_canonicalization_passes == "sroa,mem2reg"
    if candidate["loop"].num_nodes:
        loop_sample = candidate
        break
assert loop_sample is not None, "No natural-loop tensor found in the cohort"
assert loop_sample["loop"].x.shape[1] == LOOP_FEATURE_SIZE
assert ("loop", "contains", "block") in loop_sample.edge_types
assert ("pragma", "applies_to", "loop") in loop_sample.edge_types
print("Validated canonicalization provenance, schema v3, loop features, and exact cohort.")

## Matched configurations, resume safety, and time-bounded training

In [ ]:
import shlex

experiments = {
    "matched_h0": "canonical_h0_scale200_seed42",
    "region_hardware": "region_hardware_scale200_seed42",
    "region_generic": "region_generic_scale200_seed42",
}
common = {
    "tensor_dir": str(TENSOR_DIR),
    "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH),
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "require_complete_split_manifest": True,
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "scale_percent": 200,
    "distributed_world_size": GPU_COUNT if USE_DDP else 1,
    "seed": SEED,
    "family_balanced_sampling": False,
    "batch_size": 8,
    "num_workers": 2,
    "worker_tmpdir": "/tmp",
    "pin_memory": True,
    "prefetch_factor": 2,
    "thread_prefetch": False,
    "precision": PRECISION,
    "gpu_telemetry_interval_ms": 1000,
    "epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "checkpoint_interval": 5,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.15,
    "use_global_features": True,
    "use_context": True,
    "context_mode": "core",
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "verbose": 2,
}
model_configs = {
    "matched_h0": {"model": "hierarchical", "architecture_label": "H0 on canonical schema-v3 tensors"},
    "region_hardware": {
        "model": "hierarchical_region", "cardinality_messages": False,
        "composition": "hardware_aligned", "critical_path_steps": 3,
        "architecture_label": "natural loops + hardware-aligned composition",
    },
    "region_generic": {
        "model": "hierarchical_region", "cardinality_messages": False,
        "composition": "generic", "critical_path_steps": 3,
        "architecture_label": "natural loops + generic composition ablation",
    },
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
_previous_root = None

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SPLIT_SHA256 = file_sha256(SPLIT_MANIFEST_PATH)

def previous_search_root():
    global _previous_root
    if PREVIOUS_RESULTS_ROOT is None:
        return None
    if _previous_root is not None:
        return _previous_root
    root = Path(PREVIOUS_RESULTS_ROOT)
    archives = sorted(root.rglob(f"{ARCHIVE_NAME}.zip"))
    if archives:
        assert len(archives) == 1, archives
        root = WORK / "previous_region_results"
        if not root.is_dir():
            shutil.unpack_archive(archives[0], root)
    _previous_root = root
    return root

def find_previous_run(name):
    root = previous_search_root()
    if root is None:
        return None
    experiment = experiments[name]
    matches = sorted(
        path.parent for path in root.rglob("notebook_resume_signature.json")
        if path.parent.name == experiment
    )
    assert len(matches) <= 1, matches
    return matches[0] if matches else None

def baseline_path(name):
    if name == "matched_h0":
        return HISTORICAL_H0_PREDICTIONS_PATH
    h0_dir = RESULTS_DIR / experiments["matched_h0"]
    if not h0_dir.exists():
        previous = find_previous_run("matched_h0")
        if previous is not None:
            shutil.copytree(previous, h0_dir)
    path = h0_dir / "predictions.csv"
    assert path.is_file(), "Region runs require completed matched_h0 predictions."
    return path

def build_config(name):
    experiment = experiments[name]
    return {
        **common, **model_configs[name], "experiment_name": experiment,
        "checkpoint_dir": str(RESULTS_DIR / experiment / "checkpoints"),
        "baseline_predictions_path": str(baseline_path(name)),
    }

def resume_signature(name, config):
    keys = [
        "model", "cardinality_messages", "composition", "critical_path_steps",
        "scale_percent", "seed", "batch_size", "precision",
        "distributed_world_size", "epochs", "patience", "learning_rate",
        "weight_decay", "hidden_dim", "num_layers", "dropout",
        "use_global_features", "use_context", "context_mode",
        "split_heads", "hurdle_heads", "loss",
    ]
    signature = {key: config.get(key) for key in keys}
    signature.update({
        "repo_ref": REPO_REF, "tensor_source_revision": TENSOR_REVISION,
        "split_manifest_sha256": SPLIT_SHA256,
        "baseline_run": "historical_h0" if name == "matched_h0" else "matched_h0",
    })
    return signature

def prepare_run(name):
    config = build_config(name)
    experiment = experiments[name]
    run_dir = RESULTS_DIR / experiment
    expected = resume_signature(name, config)
    previous = find_previous_run(name)
    if not run_dir.exists() and previous is not None:
        prior = json.loads((previous / "notebook_resume_signature.json").read_text())
        assert prior == expected, f"Refusing incompatible resume for {experiment}"
        shutil.copytree(previous, run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    signature_path = run_dir / "notebook_resume_signature.json"
    if signature_path.is_file():
        assert json.loads(signature_path.read_text()) == expected
    signature_path.write_text(json.dumps(expected, indent=2))
    backup = Path(config["checkpoint_dir"]) / f"{experiment}_backup.pt"
    if backup.is_file():
        config["resume_checkpoint_path"] = str(backup)
    config_path = CONFIG_DIR / f"{experiment}.json"
    config_path.write_text(json.dumps(config, indent=2))
    return config, config_path

TRAIN_SCRIPT = REPO_DIR / "scripts/train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/tmp/ll-hls4ml-matplotlib"

def base_training_command(config_path):
    if USE_DDP and GPU_COUNT > 1:
        return [
            sys.executable, "-m", "torch.distributed.run", "--standalone",
            f"--nproc_per_node={GPU_COUNT}", str(TRAIN_SCRIPT), "--config", str(config_path),
        ]
    return [sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path)]

def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command, cwd=REPO_DIR, env=run_environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()

def package_results():
    archive = Path(shutil.make_archive(str(WORK / ARCHIVE_NAME), "zip", root_dir=RESULTS_DIR))
    print("Updated result archive:", archive)
    return archive

def run_experiment(name):
    if name not in ACTIVE_RUNS:
        print("Skipping", name)
        return
    config, config_path = prepare_run(name)
    experiment = experiments[name]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        if summary.get("resolved_config", {}).get("evaluation_checkpoint_path") is None:
            print(experiment, "already completed normally")
            package_results()
            return
    command = [
        "timeout", "--signal=INT", "--kill-after=5m", TRAIN_BUDGETS[name],
        *base_training_command(config_path),
    ]
    try:
        started = time.time()
        return_code = run_and_stream(command, run_dir / "training.log")
        print(experiment, "return code", return_code, "wall seconds", round(time.time() - started, 1))
        if return_code == 0 and summary_path.is_file():
            return
        best = Path(config["checkpoint_dir"]) / f"{experiment}_checkpoint.pt"
        assert best.is_file(), "Time limit expired before the first validation checkpoint."
        evaluation = [
            sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path),
            "--evaluate-checkpoint", str(best),
        ]
        assert run_and_stream(evaluation, run_dir / "evaluation.log") == 0
    finally:
        package_results()

## Primary gate: matched H0, then the full region model

In [ ]:
run_experiment("matched_h0")
run_experiment("region_hardware")

## Conditional ablation

Add `region_generic` to `ACTIVE_RUNS` only after the primary gate. It removes hardware-aligned pooling/path composition while retaining the loop hierarchy. This is the highest-value third run if the bundle gains several SMAPE points.

In [ ]:
run_experiment("region_generic")

## Paired statistical comparison

In [ ]:
import numpy as np
import pandas as pd

TARGETS = ("lut", "ff", "dsp", "bram", "cycles_max", "interval_max")

def sample_smape(frame):
    truth = frame[[f"target_{target}" for target in TARGETS]].to_numpy(float)
    prediction = frame[[f"prediction_{target}" for target in TARGETS]].to_numpy(float)
    return np.mean(200 * np.abs(prediction - truth) / (np.abs(prediction) + np.abs(truth) + 1.0), axis=1)

def score_series(path, split):
    frame = pd.read_csv(path)
    frame = frame[frame["split"] == split].copy()
    score = pd.Series(sample_smape(frame), index=frame["tensor_path"], name=Path(path).parent.name)
    assert not score.index.duplicated().any()
    return score

def paired_improvement(baseline_path, candidate_path, split, draws=10000):
    baseline = score_series(baseline_path, split)
    candidate = score_series(candidate_path, split)
    paired = pd.concat([baseline.rename("baseline"), candidate.rename("candidate")], axis=1, join="inner").dropna()
    assert len(paired) == expected_sizes[split], (split, len(paired), expected_sizes[split])
    improvement = (paired["baseline"] - paired["candidate"]).to_numpy()
    rng = np.random.default_rng(20260819)
    boot = np.fromiter(
        (rng.choice(improvement, size=len(improvement), replace=True).mean() for _ in range(draws)),
        dtype=float, count=draws,
    )
    return {
        "split": split, "n": len(paired),
        "baseline_smape": paired["baseline"].mean(),
        "candidate_smape": paired["candidate"].mean(),
        "smape_improvement": improvement.mean(),
        "ci95_low": np.quantile(boot, 0.025),
        "ci95_high": np.quantile(boot, 0.975),
        "sample_win_fraction": np.mean(improvement > 0),
    }

historical = HISTORICAL_H0_PREDICTIONS_PATH
prediction_paths = {
    name: RESULTS_DIR / experiment / "predictions.csv"
    for name, experiment in experiments.items()
    if (RESULTS_DIR / experiment / "predictions.csv").is_file()
}
rows = []
if "matched_h0" in prediction_paths:
    for split in ("test", "exemplar"):
        rows.append({
            "comparison": "historical_h0 -> canonical_h0",
            **paired_improvement(historical, prediction_paths["matched_h0"], split),
        })
for name in ("region_hardware", "region_generic"):
    if name in prediction_paths and "matched_h0" in prediction_paths:
        for split in ("test", "exemplar"):
            rows.append({
                "comparison": f"canonical_h0 -> {name}",
                **paired_improvement(prediction_paths["matched_h0"], prediction_paths[name], split),
            })
comparison = pd.DataFrame(rows)
display(comparison)
if not comparison.empty:
    comparison.to_csv(RESULTS_DIR / "paired_region_comparison.csv", index=False)
package_results()

## Decision rule

Treat the full test cohort as primary and exemplar as the robustness check. `historical_h0 -> canonical_h0` estimates the representation/canonicalization contribution. `canonical_h0 -> region_hardware` is the matched architectural bundle. Positive `smape_improvement` is better. Below 2 points, stop spending GPU on this line. Between 2 and 5 is scientifically interesting but not enough for the stated competitiveness target. At 5+ points with a positive paired interval and no severe family/target regression, run the generic ablation and then seeds 43–44. Parameter counts, structural slices, per-target metrics, predictions, and telemetry are already written by the training script.